In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('train.txt', sep = ';', header = None, names = ['text', 'emotion'])
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [3]:
df.emotion.value_counts()

emotion
joy         5362
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64

In [4]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [5]:
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# df['emotion'] = le.fit_transform(df['emotion'])
# df.head()

In [6]:
unique_emotion = df.emotion.unique()
empty_num = {}
z = 0
for i in unique_emotion:
    empty_num[i] = z
    z += 1


In [7]:
df['emotion'] = df['emotion'].map(empty_num)
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [8]:
df['text'] = df['text'].str.lower()
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [9]:
import re
df['text'] = df['text'].apply(lambda x: re.sub('[^\w\s]', '', x))   #punctuation
df['text'] = df['text'].apply(lambda x: re.sub('\d+', '', x))       #numbers
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [10]:
import emoji

df['text'] = df['text'].apply(emoji.demojize)
df.head()


,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [11]:
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /Users/apple/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/apple/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/apple/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
stop_words = set(stopwords.words('english'))
len(stop_words)

198

In [13]:
df.loc[0].text


'i didnt feel humiliated'

In [14]:
def remove_stopwords(text):
    words = word_tokenize(text)
    filtered_words = []
    for i in words:
        if i not in stop_words:
            filtered_words.append(i)       
    return ' '.join(filtered_words)

In [15]:
df['text'] = df['text'].apply(remove_stopwords)
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [16]:
from sklearn.model_selection import train_test_split

X = df['text']
y = df['emotion']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [17]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

cv = CountVectorizer()
X_train_cv = cv.fit_transform(X_train)
X_test_cv = cv.transform(X_test)

tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# MultinomialNB

In [18]:
from sklearn.naive_bayes import MultinomialNB
model_NB1 = MultinomialNB()
model_NB1.fit(X_train_cv, y_train)

MultinomialNB()

In [19]:
model_NB2 = MultinomialNB()
model_NB2.fit(X_train_tfidf, y_train)

MultinomialNB()

In [20]:
from sklearn.metrics import accuracy_score

y_pred_NB1 = model_NB1.predict(X_test_cv)
accuracy_score(y_test, y_pred_NB1)

0.7678125

In [21]:
y_pred_NB2 = model_NB2.predict(X_test_tfidf)
accuracy_score(y_test, y_pred_NB2)

0.6609375

# LogisticRegression


In [22]:
from sklearn.linear_model import LogisticRegression
model_LR1 = LogisticRegression(max_iter=100)
model_LR1.fit(X_train_cv, y_train)

LogisticRegression()

In [23]:
model_LR2 = LogisticRegression(max_iter=100)
model_LR2.fit(X_train_tfidf, y_train)

LogisticRegression()

In [24]:
print(model_LR1.n_iter_)
print(model_LR2.n_iter_)

[85]
[90]


In [25]:
y_pred_LR1 = model_LR1.predict(X_test_cv)
accuracy_score(y_test, y_pred_LR1)

0.88875

In [26]:
y_pred_LR2 = model_LR2.predict(X_test_tfidf)
accuracy_score(y_test, y_pred_LR2)

0.8615625

# using Joblib

In [27]:
import joblib
joblib.dump(model_LR1, 'LR1.joblib')
joblib.dump(cv, "Count_Vectorizer.joblib")


['Count_Vectorizer.joblib']